# Registration Point Debug Visualizer

Visualize, per frame:
- tracked + visible points
- tentative (not-yet-valid) visible points
- points used in registration
- registration inliers

It reads `meta_data/meta_data.npz` and overlays points on `with_gt` / `output_images`.


In [ ]:

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as iio

try:
    import pandas as pd
    HAS_PANDAS = True
except Exception:
    HAS_PANDAS = False

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

# -----------------------------
# Edit these paths if needed
# -----------------------------
RUN_DIR = Path('/home/justin/code/point-to-pose/results/ho3d_all/AP10')
NPZ_PATH = RUN_DIR / 'meta_data' / 'meta_data.npz'

IMAGE_VIEW_DEFAULT = 'with_gt'  # 'with_gt' or 'output_images'

assert NPZ_PATH.exists(), f'Missing npz: {NPZ_PATH}'
data = np.load(NPZ_PATH, allow_pickle=True)

print('Loaded:', NPZ_PATH)
print('Num fields:', len(data.files))
print('Has pandas:', HAS_PANDAS)
print('Has widgets:', HAS_WIDGETS)


In [ ]:

# -----------------------------
# Helpers for packed ragged fields
# -----------------------------
def has_packed_key(base: str) -> bool:
    return (
        f"{base}_data" in data.files
        and f"{base}_offsets" in data.files
        and f"{base}_lengths" in data.files
    )

def n_rows() -> int:
    if 'frame_id' in data.files:
        return len(data['frame_id'])
    for k in data.files:
        if k.endswith('_lengths'):
            return len(data[k])
    raise RuntimeError('Could not infer number of rows.')

N_ROWS = n_rows()

def get_row(base: str, i: int):
    if has_packed_key(base):
        d = data[f"{base}_data"]
        o = data[f"{base}_offsets"]
        l = data[f"{base}_lengths"]
        start = int(o[i])
        length = int(l[i])
        return d[start:start+length]

    if base in data.files:
        arr = data[base]
        if len(arr) <= i:
            return None
        v = arr[i]
        if isinstance(v, np.ndarray):
            return v
        return v

    return None

def to_scalar(v, default=np.nan):
    if v is None:
        return default
    try:
        if isinstance(v, np.ndarray):
            if v.size == 0:
                return default
            if v.size == 1:
                return float(v.reshape(-1)[0])
            return default
        return float(v)
    except Exception:
        return default

def as_1d(a, dtype=None):
    if a is None:
        out = np.empty((0,))
    else:
        out = np.asarray(a).reshape(-1)
    if dtype is not None:
        try:
            out = out.astype(dtype)
        except Exception:
            pass
    return out

def as_2d_xy(a):
    if a is None:
        return np.empty((0, 2), dtype=np.float32)
    arr = np.asarray(a)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 2) if arr.size % 2 == 0 else np.empty((0,2))
    if arr.shape[-1] != 2:
        return np.empty((0, 2), dtype=np.float32)
    return arr.astype(np.float32)

frame_ids = np.array([
    int(to_scalar(get_row('frame_id', i), default=i)) for i in range(N_ROWS)
], dtype=np.int64)

frame_to_row = {}
for i, fid in enumerate(frame_ids.tolist()):
    if fid not in frame_to_row:
        frame_to_row[fid] = i

print('Rows:', N_ROWS)
print('Frame id range:', int(frame_ids.min()), '->', int(frame_ids.max()))
print('Unique frame ids:', len(frame_to_row))


In [ ]:

# -----------------------------
# Core reconstruction for one frame/row
# -----------------------------
def load_frame_image(frame_id: int, view: str = IMAGE_VIEW_DEFAULT):
    assert view in ('with_gt', 'output_images')
    img_path = RUN_DIR / view / f'frame_{frame_id:06d}.png'
    if img_path.exists():
        return iio.imread(img_path), img_path
    return None, img_path

def build_tid_to_objrow(i: int):
    obj_idx = as_1d(get_row('extract_obj_idx', i), dtype=np.int64)
    obj_rows = as_1d(get_row('extract_obj_rows', i), dtype=np.int64)
    if obj_rows.size == obj_idx.size and obj_rows.size > 0:
        return {int(t): int(r) for t, r in zip(obj_idx, obj_rows)}
    return {int(t): j for j, t in enumerate(obj_idx.tolist())}

def classify_for_row(i: int):
    track2d = as_2d_xy(get_row('track2d', i))
    visibles = as_1d(get_row('visibles', i), dtype=bool)
    valid_depth = as_1d(get_row('valid', i), dtype=bool)

    n_tracks = track2d.shape[0]
    if visibles.size != n_tracks:
        visibles = np.zeros((n_tracks,), dtype=bool)
    if valid_depth.size != n_tracks:
        valid_depth = np.zeros((n_tracks,), dtype=bool)

    tracked_visible_mask = visibles
    tracked_visible_xy = track2d[tracked_visible_mask] if n_tracks else np.empty((0,2), dtype=np.float32)

    obj_idx = as_1d(get_row('extract_obj_idx', i), dtype=np.int64)
    vis_obj = as_1d(get_row('extract_vis_obj_mask', i), dtype=bool)
    valid_kp_obj = as_1d(get_row('extract_valid_kp_mask', i), dtype=bool)

    if vis_obj.size != obj_idx.size:
        vis_obj = np.zeros((obj_idx.size,), dtype=bool)
    if valid_kp_obj.size != obj_idx.size:
        valid_kp_obj = np.zeros((obj_idx.size,), dtype=bool)

    tentative_visible_local = vis_obj & (~valid_kp_obj)
    tid_tent_visible = obj_idx[tentative_visible_local]
    tid_tent_visible = tid_tent_visible[(tid_tent_visible >= 0) & (tid_tent_visible < n_tracks)]
    tentative_visible_xy = track2d[tid_tent_visible] if tid_tent_visible.size else np.empty((0,2), dtype=np.float32)

    reg_valid_idx = as_1d(get_row('reg_valid_idx', i), dtype=np.int64)
    reg_valid_idx = reg_valid_idx[(reg_valid_idx >= 0) & (reg_valid_idx < n_tracks)]
    reg_used_xy = track2d[reg_valid_idx] if reg_valid_idx.size else np.empty((0,2), dtype=np.float32)

    reg_inliers = as_1d(get_row('reg_inliers', i), dtype=bool)
    if reg_inliers.size != reg_valid_idx.size:
        reg_inliers = np.zeros((reg_valid_idx.size,), dtype=bool)
    inlier_tids = reg_valid_idx[reg_inliers]
    inlier_xy = track2d[inlier_tids] if inlier_tids.size else np.empty((0,2), dtype=np.float32)

    tid2row = build_tid_to_objrow(i)
    obj_valid = as_1d(get_row('obj_valid', i), dtype=bool)
    is_tent_used = np.zeros((reg_valid_idx.size,), dtype=bool)
    for k, tid in enumerate(reg_valid_idx.tolist()):
        row = tid2row.get(int(tid), -1)
        if 0 <= row < obj_valid.size:
            is_tent_used[k] = (not bool(obj_valid[row]))
        else:
            is_tent_used[k] = False

    used_tentative_xy = track2d[reg_valid_idx[is_tent_used]] if reg_valid_idx.size else np.empty((0,2), dtype=np.float32)
    used_confirmed_xy = track2d[reg_valid_idx[~is_tent_used]] if reg_valid_idx.size else np.empty((0,2), dtype=np.float32)

    best_cluster_idx = int(to_scalar(get_row('reg_best_cluster_idx', i), default=-999))
    residuals = as_1d(get_row('reg_residuals', i), dtype=float)
    mean_res = float(np.nan)
    if residuals.size > 0:
        if residuals.size == reg_inliers.size and np.any(reg_inliers):
            mean_res = float(np.mean(residuals[reg_inliers]))
        else:
            mean_res = float(np.mean(residuals))

    fb_used = bool(to_scalar(get_row('extract_tentative_fallback_used', i), default=0.0))
    fb_add = int(to_scalar(get_row('extract_tentative_added_count', i), default=0))
    fb_pool = int(to_scalar(get_row('extract_tentative_pool_count', i), default=0))
    fb_min = int(to_scalar(get_row('extract_tentative_fallback_min_valid_points', i), default=0))

    return {
        'track2d': track2d,
        'tracked_visible_xy': tracked_visible_xy,
        'tentative_visible_xy': tentative_visible_xy,
        'reg_used_xy': reg_used_xy,
        'inlier_xy': inlier_xy,
        'used_tentative_xy': used_tentative_xy,
        'used_confirmed_xy': used_confirmed_xy,
        'n_tracks': int(n_tracks),
        'n_visible': int(tracked_visible_xy.shape[0]),
        'n_tent_visible': int(tentative_visible_xy.shape[0]),
        'n_reg_used': int(reg_valid_idx.size),
        'n_inliers': int(np.sum(reg_inliers)),
        'best_cluster_idx': best_cluster_idx,
        'mean_res': mean_res,
        'fb_used': fb_used,
        'fb_add': fb_add,
        'fb_pool': fb_pool,
        'fb_min': fb_min,
    }


In [ ]:

# -----------------------------
# Plot one frame
# -----------------------------
def plot_frame_debug(frame_id: int, view: str = IMAGE_VIEW_DEFAULT, figsize=(12, 8)):
    if frame_id not in frame_to_row:
        raise KeyError(f'Frame {frame_id} not found in logs.')

    i = frame_to_row[frame_id]
    rec = classify_for_row(i)
    img, img_path = load_frame_image(frame_id, view=view)

    fig, ax = plt.subplots(1, 1, figsize=figsize)
    if img is not None:
        ax.imshow(img)
    else:
        ax.set_title(f'Image not found: {img_path}')
        ax.set_xlim(0, 640)
        ax.set_ylim(480, 0)

    if rec['tracked_visible_xy'].size:
        ax.scatter(
            rec['tracked_visible_xy'][:, 0],
            rec['tracked_visible_xy'][:, 1],
            s=8,
            c='lightgray',
            alpha=0.40,
            label=f"tracked+visible ({rec['n_visible']})",
            edgecolors='none',
        )

    if rec['tentative_visible_xy'].size:
        ax.scatter(
            rec['tentative_visible_xy'][:, 0],
            rec['tentative_visible_xy'][:, 1],
            s=22,
            facecolors='none',
            edgecolors='gold',
            linewidths=1.1,
            label=f"tentative visible ({rec['n_tent_visible']})",
        )

    if rec['used_confirmed_xy'].size:
        ax.scatter(
            rec['used_confirmed_xy'][:, 0],
            rec['used_confirmed_xy'][:, 1],
            s=30,
            c='deepskyblue',
            marker='x',
            linewidths=1.2,
            label=f"used (confirmed) ({rec['used_confirmed_xy'].shape[0]})",
        )
    if rec['used_tentative_xy'].size:
        ax.scatter(
            rec['used_tentative_xy'][:, 0],
            rec['used_tentative_xy'][:, 1],
            s=32,
            c='magenta',
            marker='x',
            linewidths=1.4,
            label=f"used (tentative) ({rec['used_tentative_xy'].shape[0]})",
        )

    if rec['inlier_xy'].size:
        ax.scatter(
            rec['inlier_xy'][:, 0],
            rec['inlier_xy'][:, 1],
            s=52,
            c='lime',
            marker='+',
            linewidths=1.5,
            label=f"inliers ({rec['n_inliers']})",
        )

    ax.set_axis_off()
    ax.legend(loc='upper right', fontsize=9, framealpha=0.8)

    txt = (
        f"frame={frame_id} | tracks={rec['n_tracks']} | visible={rec['n_visible']} | "
        f"reg_used={rec['n_reg_used']} | inliers={rec['n_inliers']} | "
        f"best_cluster_idx={rec['best_cluster_idx']} | mean_res={rec['mean_res']:.5f}\n"
        f"fallback_used={rec['fb_used']} | fb_add={rec['fb_add']} | fb_pool={rec['fb_pool']} | fb_min={rec['fb_min']}"
    )
    ax.set_title(txt, fontsize=10)
    plt.tight_layout()
    plt.show()

    return rec

_ = plot_frame_debug(int(frame_ids[0]), view=IMAGE_VIEW_DEFAULT)


In [ ]:

# -----------------------------
# Build diagnostics rows
# -----------------------------
diag_rows = []
for fid, i in frame_to_row.items():
    rec = classify_for_row(i)
    diag_rows.append({
        'frame_id': int(fid),
        'n_tracks': rec['n_tracks'],
        'n_visible': rec['n_visible'],
        'n_tentative_visible': rec['n_tent_visible'],
        'n_reg_used': rec['n_reg_used'],
        'n_inliers': rec['n_inliers'],
        'best_cluster_idx': rec['best_cluster_idx'],
        'mean_res': rec['mean_res'],
        'fallback_used': rec['fb_used'],
        'fallback_added': rec['fb_add'],
        'fallback_pool': rec['fb_pool'],
        'fallback_min': rec['fb_min'],
    })

diag_rows = sorted(diag_rows, key=lambda r: r['frame_id'])

if HAS_PANDAS:
    diag_df = pd.DataFrame(diag_rows)
    display(diag_df.head())
else:
    print('First 10 rows:')
    for r in diag_rows[:10]:
        print(r)


In [ ]:

# Frames where registration had candidates used but got 0 inliers
zero_inlier_rows = [r for r in diag_rows if (r['n_reg_used'] > 0 and r['n_inliers'] == 0)]
print('zero-inlier frames:', len(zero_inlier_rows))

if HAS_PANDAS:
    display(pd.DataFrame(zero_inlier_rows).head(30))
else:
    for r in zero_inlier_rows[:30]:
        print(r)


In [ ]:

# -----------------------------
# Interactive viewer
# -----------------------------
if HAS_WIDGETS:
    frame_options = sorted(frame_to_row.keys())

    def _show(frame_id, view):
        plot_frame_debug(int(frame_id), view=view)

    widgets.interact(
        _show,
        frame_id=widgets.SelectionSlider(
            options=frame_options,
            value=frame_options[0],
            description='frame',
            continuous_update=False,
            layout=widgets.Layout(width='900px'),
        ),
        view=widgets.ToggleButtons(
            options=['with_gt', 'output_images'],
            value=IMAGE_VIEW_DEFAULT,
            description='view',
        ),
    )
else:
    print('ipywidgets not available. Use plot_frame_debug(frame_id).')


In [ ]:

# -----------------------------
# Batch visualize top-K failure frames (zero inliers)
# -----------------------------
def plot_failure_gallery(k=12, view=IMAGE_VIEW_DEFAULT):
    bad = [r for r in diag_rows if (r['n_reg_used'] > 0 and r['n_inliers'] == 0)]
    if len(bad) == 0:
        print('No zero-inlier frames found.')
        return

    bad = sorted(bad, key=lambda r: (-r['n_reg_used'], -r['fallback_added']))[:k]
    frames = [int(r['frame_id']) for r in bad]

    n = len(frames)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.5 * rows))
    axes = np.asarray(axes).reshape(-1)

    for ax, fid in zip(axes, frames):
        i = frame_to_row[int(fid)]
        rec = classify_for_row(i)
        img, _ = load_frame_image(int(fid), view=view)

        if img is not None:
            ax.imshow(img)
        else:
            ax.set_xlim(0, 640)
            ax.set_ylim(480, 0)

        if rec['tracked_visible_xy'].size:
            ax.scatter(rec['tracked_visible_xy'][:,0], rec['tracked_visible_xy'][:,1], s=6, c='lightgray', alpha=0.35, edgecolors='none')
        if rec['reg_used_xy'].size:
            ax.scatter(rec['reg_used_xy'][:,0], rec['reg_used_xy'][:,1], s=24, c='deepskyblue', marker='x', linewidths=1.1)
        if rec['inlier_xy'].size:
            ax.scatter(rec['inlier_xy'][:,0], rec['inlier_xy'][:,1], s=42, c='lime', marker='+', linewidths=1.3)

        ax.set_axis_off()
        ax.set_title(
            f"f={fid} used={rec['n_reg_used']} inl={rec['n_inliers']} "
            f"fb_add={rec['fb_add']} best={rec['best_cluster_idx']}",
            fontsize=9,
        )

    for ax in axes[n:]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

plot_failure_gallery(k=12, view=IMAGE_VIEW_DEFAULT)
